# Explore Vietnamese Legal Dataset

Clean, portable exploration notebook for the SME legal corpus. Generated files are written under `data/` or `artifacts/` so the workflow is restartable.


## 1. Setup

Paths are resolved relative to the `vietnamese-legal-rag` project root; no machine-specific Windows or Colab paths are required.


In [1]:
import os
os.chdir("Road2AI_ApplePie/vietnamese-legal-rag")

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("..").resolve()

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
REPORT_DIR = PROJECT_ROOT / "artifacts" / "reports"

METADATA_PATH = DATA_DIR / "metadata.jsonl"
CONTENT_PATH = DATA_DIR / "content.jsonl"
RELATIONSHIPS_PATH = DATA_DIR / "relationships.jsonl"
CONFIG_PATH = PROJECT_ROOT / "configs" / "smecorpus.yaml"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

print(f"Project root: {PROJECT_ROOT}")


Project root: D:\Personal Project\Road2AI_Apple_Pie\Road2AI_ApplePie\vietnamese-legal-rag


## 2. Load metadata


In [3]:
def read_jsonl(path: Path) -> pd.DataFrame:
    """Read a JSONL file and fail with a clear message if it is missing."""
    if not path.exists():
        raise FileNotFoundError(f"Missing expected dataset file: {path}")
    return pd.read_json(path, lines=True)

metadata = read_jsonl(METADATA_PATH)
print(f"Metadata rows: {len(metadata):,}")
print(f"Columns: {', '.join(metadata.columns)}")
display(metadata.head())


Metadata rows: 153,420
Columns: id, title, so_ky_hieu, ngay_ban_hanh, loai_van_ban, ngay_co_hieu_luc, ngay_het_hieu_luc, nguon_thu_thap, ngay_dang_cong_bao, nganh, linh_vuc, co_quan_ban_hanh, chuc_danh, nguoi_ky, pham_vi, thong_tin_ap_dung, tinh_trang_hieu_luc


,id,title,so_ky_hieu,ngay_ban_hanh,loai_van_ban,ngay_co_hieu_luc,ngay_het_hieu_luc,nguon_thu_thap,ngay_dang_cong_bao,nganh,linh_vuc,co_quan_ban_hanh,chuc_danh,nguoi_ky,pham_vi,thong_tin_ap_dung,tinh_trang_hieu_luc
0,72,Sắc lệnh quy định liên hệ giữa UBKCHC và các cơ quan chuyên môn,103/SL,05/06/1950,Sắc lệnh,20/06/1950,NaN,Công báo số 7/1950;,...,NaN,NaN,Chủ tịch nước,Chủ tịch nước,Hồ Chí Minh,NaN,NaN,Hết hiệu lực toàn bộ
1,4260,Về việc chuyển giao nhiệm vụ quản lý Nhà nước về đất lâm nghiệp từ Sở Nông nghiệp và Phát triển Nông thôn và Chi cục...,87/1999/QĐ-UB,12/07/1999,Quyết định,12/07/1999,NaN,STP tỉnh Lâm Đồng;,...,NaN,NaN,UBND tỉnh Lâm Đồng,Q.Chủ tịch,Đặng Đức Lợi,Lâm Đồng,NaN,Còn hiệu lực
2,4280,Về viẹc quản lý và quy định thống nhất phí chợ trên địa bàn tỉnh Lâm Đồng,157/1999/QĐ-UB,19/11/1999,Quyết định,19/11/1999,NaN,STP tỉnh Lâm Đồng;,...,Tài chính,NaN,UBND tỉnh Lâm Đồng,Phó Chủ tịch,Trương Thành Trung,Lâm Đồng,NaN,Còn hiệu lực
3,4263,"Về việc xử lý đối với quỹ nhà thuộc quyền sở hữu Nhà nước do các cơ quan, đơn vị đã tự bố trí cho cán bộ, công nhân ...",163/1999/QĐ-UB,30/11/1999,Quyết định,30/11/1999,NaN,STP tỉnh Lâm Đồng;,...,Xây dựng,NaN,UBND tỉnh Lâm Đồng,Q.Chủ tịch,Đặng Đức Lợi,Lâm Đồng,NaN,Còn hiệu lực
4,77,Sắc lệnh bãi bỏ Sắc lệnh 106 ngày 09-09-1949 về phí cấp hàng tháng của các các vị trong Ban thường trực Quốc hội,83/SL,22/05/1950,Sắc lệnh,01/05/1950,NaN,Hồ sơ số Q009-H002A/LTQG;,...,NaN,NaN,Chủ tịch nước,Chủ tịch nước,Hồ Chí Minh,NaN,NaN,Hết hiệu lực toàn bộ


## 3. Explore legal sectors (`linh_vuc`)


In [6]:
sector_counts = (
    metadata["linh_vuc"]
    .fillna("")
    .astype(str)
    .value_counts()
    .rename_axis("linh_vuc")
    .reset_index(name="document_count")
)

unique_sector_path = REPORT_DIR / "linh_vuc_unique.txt"
unique_sector_path.write_text(
    "\n".join(sector_counts["linh_vuc"].sort_values()) + "\n",
    encoding="utf-8",
)

print(f"Unique sectors: {len(sector_counts):,}")
print(f"Saved unique sector list to: {unique_sector_path.relative_to(PROJECT_ROOT)}")
display(sector_counts.head(30))


Unique sectors: 1,675
Saved unique sector list to: artifacts\reports\linh_vuc_unique.txt


,linh_vuc,document_count
0,,107232
1,Đất đai,2486
2,"Quản lý thuế, phí, lệ phí và thu khác của ngân sách nhà nước",2355
3,"Quản lý thuế, phí và lệ phí",1693
4,Lĩnh vực giá,1630
5,Quản lý ngân sách,1220
6,Tổ chức- Biên chế,1183
7,Quản lý ngân sách nhà nước,1063
8,Ngân sách nhà nước,947
9,Chính quyền địa phương,850


## 4. Apply the project SME filter

The filtering logic is centralized in `src/vlr/preprocessing/sme_filter.py` and configured by `configs/smecorpus.yaml`, keeping this notebook aligned with the preprocessing pipeline.


In [7]:
import pandas as pd
from datasets import load_dataset

# --- ĐỊNH NGHĨA TIÊU CHÍ ---
exclude_linh_vuc = [
    "quốc phòng", "an ninh quốc gia", "ngoại giao", "tổ chức bộ máy nhà nước",
    "cán bộ", "công chức", "viên chức", "bầu cử", "thi đua", "khen thưởng",
    "tôn giáo", "dân tộc"
]

include_linh_vuc_truc_tiep = [
    "doanh nghiệp", "đầu tư", "thương mại", "dân sự", "hành chính", "trọng tài thương mại",
    "thuế", "phí", "lệ phí", "kế toán", "kiểm toán", "lao động", "tiền lương",
    "bảo hiểm", "sở hữu trí tuệ", "cạnh tranh", "phá sản", "ngân hàng",
    "tín dụng", "hải quan", "xuất nhập khẩu", "đấu thầu", "giao dịch điện tử",
    "công nghệ thông tin", "đất đai", "xây dựng", "môi trường", "an toàn thực phẩm", "tiêu chuẩn", "chất lượng sản phẩm"
]

conditional_others = [
    "tố tụng", "thi hành án", "nông nghiệp", "y tế",
    "giáo dục", "du lịch", "khoa học công nghệ", "thông tin truyền thông", "hình sự",

]

def apply_legal_filters(row):
    # Chuyển về chữ thường để loại bỏ sai lệch do in hoa/in thường
    sectors = str(row['linh_vuc']).lower()
    title = str(row['title']).lower()

    # BƯỚC 1: QUÉT NHÓM EXCLUDE (QUYỀN LỰC CAO NHẤT)
    # Chỉ cần dính 1 tag thuộc nhóm cấm -> Bỏ qua toàn bộ văn bản
    if any(excl in sectors for excl in exclude_linh_vuc):
        return False

    # BƯỚC 2: QUÉT CÁC LĨNH VỰC CÓ ĐIỀU KIỆN RÀNG BUỘC (Conditional & Restricted Includes)

    # 2.1 Hình sự: Chỉ lấy tội phạm kinh tế & trách nhiệm pháp nhân
    if "hình sự" in sectors:
        if any(kw in title for kw in ["kinh tế", "pháp nhân", "thương mại", "doanh nghiệp", "trốn thuế", "lừa đảo"]):
            return True

    # 2.2 Hành chính: Xoáy sâu vào xử phạt (Vì bạn note "doanh nghiệp cực kỳ quan tâm mức phạt")
    if "hành chính" in sectors:
        if "xử phạt" in title or "vi phạm" in title:
            return True

    # 2.3 An toàn thực phẩm: Chỉ lấy thủ tục cấp phép, điều kiện kinh doanh
    if "an toàn thực phẩm" in sectors:
        if any(kw in title for kw in ["cấp phép", "giấy phép", "điều kiện", "kinh doanh", "cơ sở"]):
            return True

    # 2.4 Tiêu chuẩn / Chất lượng: Chỉ lấy công bố hợp chuẩn/hợp quy
    if "tiêu chuẩn" in sectors or "chất lượng sản phẩm" in sectors:
        if any(kw in title for kw in ["hợp chuẩn", "hợp quy", "công bố", "kiểm định"]):
            return True

    # BƯỚC 3: QUÉT NHÓM INCLUDE TRỰC TIẾP (Cốt lõi)
    # Sau khi đã lọc Exclude ở bước 1, nếu thuộc danh sách này -> Giữ lại
    if any(incl in sectors for incl in include_linh_vuc_truc_tiep):
        return True

    # BƯỚC 4: XỬ LÝ NHÓM CONDITIONAL CÒN LẠI
    # (Y tế, Giáo dục, Nông nghiệp...)
    if any(cond in sectors for cond in conditional_others):
        # Mặc định giữ lại do đã qua màng lọc Exclude.
        # Nếu bạn muốn ép các văn bản này phải liên quan tới doanh nghiệp,
        # có thể mở comment dòng dưới và return False nếu không chứa từ khóa kinh tế.
        # if not any(kw in title for kw in ["kinh doanh", "tư thục", "công ty", "doanh nghiệp"]): return False
        return True

    # Nếu không thỏa mãn bất kỳ điều kiện nào -> Loại bỏ
    return False

# 2. Chạy bộ lọc
print(f"Số văn bản gốc: {len(metadata):,}")
df_filtered = metadata[metadata.apply(apply_legal_filters, axis=1)]
print(f"Số văn bản sau khi lọc: {len(df_filtered):,}")

# 3. Xuất file hoặc tiếp tục xử lý
# df_filtered.to_parquet("vietnamese_corporate_legal_corpus.parquet")

Số văn bản gốc: 153,420
Số văn bản sau khi lọc: 17,071


## 5. Inspect unecessary sectors


In [9]:
EXCLUDE_LINH_VUC = "D:\\Personal Project\\Road2AI_Apple_Pie\\Road2AI_ApplePie\\vietnamese-legal-rag\\data\\exclude_linh_vuc.txt"

In [12]:
import os

# Tên các file đầu vào và đầu ra
file_exclude = EXCLUDE_LINH_VUC
file_input = "D:\\Personal Project\\Road2AI_Apple_Pie\\Road2AI_ApplePie\\vietnamese-legal-rag\\data\\linh_vuc_filtered_unique.txt"
file_output = "D:\\Personal Project\Road2AI_Apple_Pie\\Road2AI_ApplePie\\vietnamese-legal-rag\\data\\final_linh_vuc.txt"
# file_removed = "/content/removed_linh_vuc.txt" # Tùy chọn: Lưu lại các dòng bị xóa để kiểm tra

def filter_txt_file():
    # 1. Đọc danh sách các từ khóa cần loại trừ
    if not os.path.exists(file_exclude):
        print(f"Lỗi: Không tìm thấy file '{file_exclude}'")
        return

    with open(file_exclude, "r", encoding="utf-8") as f:
        # Lấy từng dòng, xóa khoảng trắng 2 đầu và chuyển thành chữ thường
        exclude_list = [line.strip().lower() for line in f if line.strip()]

    print(f"Đã tải {len(exclude_list)} từ khóa loại trừ từ '{file_exclude}'.")

    # 2. Đọc file cần lọc
    if not os.path.exists(file_input):
        print(f"Lỗi: Không tìm thấy file '{file_input}'")
        return

    kept_lines = []
    removed_lines = []

    with open(file_input, "r", encoding="utf-8") as f:
        for line in f:
            original_line = line.strip()
            if not original_line:
                continue # Bỏ qua dòng trống

            # Chuyển dòng hiện tại về chữ thường để so sánh không phân biệt hoa/thường
            line_lower = original_line.lower()

            # Kiểm tra xem dòng này có chứa BẤT KỲ từ khóa cấm nào không
            is_excluded = any(excl in line_lower for excl in exclude_list)

            if is_excluded:
                removed_lines.append(original_line)
            else:
                kept_lines.append(original_line)

    # 3. Ghi các lĩnh vực đã được giữ lại ra file mới
    with open(file_output, "w", encoding="utf-8") as f:
        for line in kept_lines:
            f.write(line + "\n")

    # 4. (Tùy chọn) Ghi các lĩnh vực bị loại ra file riêng để bạn dễ dò lại
    # with open(file_removed, "w", encoding="utf-8") as f:
    #     for line in removed_lines:
    #         f.write(line + "\n")

    # 5. In báo cáo thống kê
    print("-" * 30)
    print("QUÁ TRÌNH LỌC HOÀN TẤT!")
    print(f"Số lĩnh vực giữ lại (Sạch): {len(kept_lines)} -> Đã lưu vào '{file_output}'")
    # print(f"Số lĩnh vực bị loại bỏ: {len(removed_lines)} -> Đã lưu vào '{file_removed}'")

# Chạy chương trình
if __name__ == "__main__":
    filter_txt_file()

Đã tải 34 từ khóa loại trừ từ 'D:\Personal Project\Road2AI_Apple_Pie\Road2AI_ApplePie\vietnamese-legal-rag\data\exclude_linh_vuc.txt'.
------------------------------
QUÁ TRÌNH LỌC HOÀN TẤT!
Số lĩnh vực giữ lại (Sạch): 477 -> Đã lưu vào 'D:\Personal Project\Road2AI_Apple_Pie\Road2AI_ApplePie\vietnamese-legal-rag\data\final_linh_vuc.txt'


## Save the data

In [15]:
# Load the final_linh_vuc from the text file
with open(f"{file_output}", 'r', encoding='utf-8') as f:
    final_linh_vuc = [line.strip() for line in f if line.strip()]

# Filter df_filtered based on the 'linh_vuc' column being in final_linh_vuc
final_df = df_filtered[df_filtered['linh_vuc'].isin(final_linh_vuc)]

print(f"Number of documents in final_df: {len(final_df):,}")
display(final_df.head())

Number of documents in final_df: 13,347


,id,title,so_ky_hieu,ngay_ban_hanh,loai_van_ban,ngay_co_hieu_luc,ngay_het_hieu_luc,nguon_thu_thap,ngay_dang_cong_bao,nganh,linh_vuc,co_quan_ban_hanh,chuc_danh,nguoi_ky,pham_vi,thong_tin_ap_dung,tinh_trang_hieu_luc
152,2253,Về việc chuyển giao nhiệm vụ quản lý nhập khẩu chính ngạch hàng tiêu dùng từ các Bộ khác sang Bộ Nội thương,104/CT,29/04/1989,Quyết định,29/04/1989,NaN,Công báo số 9/1989;,15/05/1989,Công Thương,Xuất nhập khẩu,Chủ tịch Hội đồng Bộ trưởng,Phó Chủ tịch,Võ Văn Kiệt,NaN,NaN,Ngưng hiệu lực
168,2249,"Về việc quản lý thống nhất xuất, nhập khẩu thuốc và nguyên liệu làm thuốc chữa cho người bệnh",113/HĐBT,09/05/1989,Quyết định,09/05/1989,NaN,Công báo số 11/1989;,15/06/1989,Công Thương,Xuất nhập khẩu,Chủ tịch Hội đồng Bộ trưởng,Phó Chủ tịch,Võ Văn Kiệt,NaN,NaN,Còn hiệu lực
215,2233,"Quy định chi tiết thi hành Nghị quyết số 134-NQ/HĐNN8 ngày 03-03-1989 của Hội đồng Nhà nước và Luật Thuế xuất khẩu, ...",54/HĐBT,27/05/1989,Nghị định,27/05/1989,NaN,Công báo số 10/1989;,31/05/1989,Công Thương,Xuất nhập khẩu,Hội đồng Bộ trưởng,Phó Chủ tịch,Võ Văn Kiệt,NaN,NaN,Hết hiệu lực toàn bộ
420,6263,"Hướng dẫn tổ chức thu, nộp, quản lý và sử dụng tiền phạt vi phạm hành chính trong vùng lãnh hải, vùng tiếp giáp lãnh...",19/2000/TTLT/BTC-BQP,14/03/2000,Thông tư liên tịch,29/03/2000,NaN,"Công báo số 14, năm 2000",15/04/2000,Quốc phòng Tài chính,Quản lý tài chính doanh nghiệp,Bộ Tài chính,Thứ trưởng,Nguyễn Văn Rinh,Toàn quốc,NaN,Hết hiệu lực toàn bộ
438,2816,Hướng dẫn về thẩm quyền và giải quyết thủ tục giải quyết những việc ly hôn của các công dân Việt Nam mà một bên ở nư...,06-TT/LN,30/12/1986,Thông tư liên tịch,14/01/1987,01/01/2001,Công báo số 2,31/01/1987,Tư pháp,Dân sự - Kinh tế,Toà án nhân dân tối cao,Phó Chánh án,Nguyễn Thị Ngọc Khanh,Toàn quốc,NaN,Hết hiệu lực toàn bộ


In [17]:
final_df.to_json("D:\\Personal Project\\Road2AI_Apple_Pie\\Road2AI_ApplePie\\vietnamese-legal-rag\\data\\final_sme_data.json")

## 6. Optional clustering visualization

Install optional dependencies only if needed:

```bash
pip install sentence-transformers umap-learn hdbscan plotly scikit-learn
```

The clustering cells are optional and can be skipped during basic data exploration.


In [ ]:
RUN_CLUSTERING = False
TEXT_COLUMN = "linh_vuc_original"
SAMPLE_SIZE = None  # Set an integer such as 5000 for faster experiments.

if RUN_CLUSTERING:
    import hdbscan
    import plotly.express as px
    import umap.umap_ as umap
    from sentence_transformers import SentenceTransformer

    cluster_df = filtered_metadata.copy()
    if SAMPLE_SIZE is not None and len(cluster_df) > SAMPLE_SIZE:
        cluster_df = cluster_df.sample(SAMPLE_SIZE, random_state=42).sort_values("lawid")

    texts = cluster_df[TEXT_COLUMN].fillna("Văn bản không có tiêu đề").astype(str).tolist()
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    embeddings = model.encode(texts, show_progress_bar=True)

    reduced_for_cluster = umap.UMAP(n_components=5, metric="cosine", random_state=42).fit_transform(embeddings)
    clusters = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5, metric="euclidean").fit_predict(reduced_for_cluster)

    reduced_2d = umap.UMAP(n_components=2, metric="cosine", random_state=42).fit_transform(embeddings)
    plot_df = pd.DataFrame({
        "lawid": cluster_df["lawid"].to_numpy(),
        "x": reduced_2d[:, 0],
        "y": reduced_2d[:, 1],
        "cluster_id": clusters,
        "cluster": [f"Cụm {c}" if c != -1 else "Nhiễu (Noise)" for c in clusters],
        "text": texts,
    })

    fig = px.scatter(
        plot_df, x="x", y="y", color="cluster",
        hover_data=["lawid", "text"],
        title="SME Legal Corpus Clusters (HDBSCAN)",
        color_discrete_sequence=px.colors.qualitative.Alphabet,
    )
    fig.update_traces(marker={"opacity": 0.7})
    fig.show()

    clustered_path = INTERIM_DIR / "sme_legal_corpus_clustered.csv"
    plot_df.to_csv(clustered_path, index=False, encoding="utf-8")
    print(f"Saved clustering output to: {clustered_path.relative_to(PROJECT_ROOT)}")
else:
    print("Set RUN_CLUSTERING = True to run optional embedding and clustering analysis.")


## 7. Optional cluster labeling

Run this after the clustering cell has created `plot_df`.


In [ ]:
if RUN_CLUSTERING and "plot_df" in globals():
    from sklearn.feature_extraction.text import TfidfVectorizer

    STOP_WORDS_VI = [
        "và", "của", "các", "về", "trong", "cho", "tại", "với", "những",
        "việc", "sự", "nhà", "nước", "thu", "khác", "hoạt", "động", "quản", "lý",
    ]

    cluster_docs = plot_df.groupby("cluster")["text"].apply(" ".join).reset_index()
    vectorizer = TfidfVectorizer(stop_words=STOP_WORDS_VI, ngram_range=(1, 3), max_features=1000)
    tfidf_matrix = vectorizer.fit_transform(cluster_docs["text"])
    feature_names = vectorizer.get_feature_names_out()

    cluster_labels = {}
    for row_idx, row in cluster_docs.iterrows():
        cluster_name = row["cluster"]
        if cluster_name == "Nhiễu (Noise)":
            cluster_labels[cluster_name] = cluster_name
            continue
        scores = tfidf_matrix.getrow(row_idx).toarray()[0]
        top_indices = scores.argsort()[-3:][::-1]
        top_phrases = [feature_names[idx].title() for idx in top_indices if scores[idx] > 0]
        cluster_labels[cluster_name] = f"{cluster_name}: {', '.join(top_phrases)}"

    plot_df["cluster_label"] = plot_df["cluster"].map(cluster_labels)
    display(pd.Series(cluster_labels, name="label").rename_axis("cluster").reset_index())
else:
    print("Run the optional clustering cell first.")
